In [ ]:
# ==========================================
# Imports
# ==========================================

import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=RuntimeWarning, module="langchain_community")
warnings.filterwarnings("ignore", category=UserWarning, module="langchain_community")

from langchain_core.tools import tool
from pprint import pprint

print("✅ All imports loaded successfully")

In [ ]:
# ==========================================
# Setup Project Root Path
# ==========================================

import sys
from pathlib import Path

# Start from the notebook directory to locate the shared loaders.
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project imports are ready")

In [ ]:
# ==========================================
# Load Configuration
# ==========================================

from src.utils.config_loader import load_config
config = load_config()

print("Configuration Loaded Successfully\n")
print(config)

In [ ]:
# ==========================================
# Load LLM
# ==========================================

from src.utils.model_loader import load_llm
llm = load_llm()

print("LLM Loaded Successfully\n")
print(type(llm))

In [ ]:
# ==========================================
# Verify LLM Connection
# ==========================================

response = llm.invoke("In one sentence, explain what LangChain is.")
print(response.content)

In [ ]:
# ==========================================
# Calculator Tool
# ==========================================

from langchain_core.tools import tool


@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression.
    Example:
    25 * 10
    100 / 4
    (10 + 5) * 2
    """

    # This demo evaluates trusted arithmetic expressions only.
    result = eval(expression)

    return str(result)


print("Tool Name :", calculator.name)
print("Tool Description :\n")
print(calculator.description)

In [ ]:
# ==========================================
# Test Calculator Tool
# ==========================================

result = calculator.invoke("125 * 8")
print("Result:", result)

Install the project dependencies from requirements.txt before running the search tool.

In [ ]:
# ==========================================
# Create Search Tool
# ==========================================

from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

print("Tool Created Successfully")
print(type(search_tool))

In [ ]:
# ==========================================
# Test Search Tool
# ==========================================

result = search_tool.invoke("What is Agentic AI?")
print(result)

In [ ]:
# ==========================================
# Create Keyword Extractor Tool
# ==========================================

from langchain_core.tools import tool


@tool
def keyword_extractor(text: str) -> str:
    """
    Extract important keywords from a given text.
    """

    prompt = f"""
    Extract the 5 most important keywords from the text below.

    Text:
    {text}

    Return only the keywords as a comma separated list.
    """

    response = llm.invoke(prompt)

    return response.content


print("Tool Name :", keyword_extractor.name)
print("\nTool Description:\n")
print(keyword_extractor.description)

In [ ]:
# ==========================================
# Test Keyword Extractor Tool
# ==========================================

result = keyword_extractor.invoke(
    """
    LangGraph is a framework for building stateful,
    multi-agent applications using large language models.
    It provides graph-based orchestration and workflow management.
    """
)

print(result)

In [ ]:
# ==========================================
# Create Research Summary Tool
# ==========================================

from langchain_core.tools import tool


@tool
def research_summary_tool(topic: str) -> str:
    """
    Generate a concise research summary for a given topic.
    """

    prompt = f"""
    You are an AI research assistant.

    Generate a concise research summary on:

    Topic: {topic}

    Requirements:
    - Maximum 150 words
    - Professional tone
    - Mention key concepts
    - Mention practical applications
    """

    response = llm.invoke(prompt)

    return response.content


print("Tool Name :", research_summary_tool.name)
print("\nTool Description:\n")
print(research_summary_tool.description)

In [ ]:
# ==========================================
# Test Research Summary Tool
# ==========================================

result = research_summary_tool.invoke("LangGraph")
print(result)

In [ ]:
# ==========================================
# Register All Tools
# ==========================================

tools = [
    calculator,
    search_tool,
    keyword_extractor,
    research_summary_tool
]

print(f"Total Tools Registered: {len(tools)}")

for tool_obj in tools:
    print("-", tool_obj.name)

In [ ]:
# ==========================================
# Create Agent Prompt
# ==========================================

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are an AI Research Assistant.

            You have access to several tools.

            Use tools whenever necessary.

            For:
            - Calculations -> use calculator
            - Web search -> use search tool
            - Keyword extraction -> use keyword_extractor
            - Research summaries -> use research_summary_tool

            Always provide clear and professional answers.
            """
        ),
        ("human", "{input}"),
        # Carry tool calls and results into the next model turn.
        ("placeholder", "{agent_scratchpad}")
    ]
)

print("Prompt Created Successfully")

In [ ]:
# ==========================================
# Create Tool Calling Agent
# ==========================================

from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor

agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

print("Tool Calling Agent Created Successfully")

In [ ]:
# ==========================================
# Test Simple Tool Calling
# ==========================================

response = agent_executor.invoke({"input": "What is 125 * 75?"})

print("\nFinal Answer:\n")
print(response["output"])

In [ ]:
response = agent_executor.invoke(
    {
        "input": """
        Search the web for the latest trends in Agentic AI
        and provide a concise summary.
        """
    }
)

print(response["output"])

In [ ]:
response = agent_executor.invoke(
    {
        "input": """
        Extract important keywords from the following text
        and then provide a concise research summary.

        LangGraph is a framework for building stateful
        multi-agent applications using LLMs.
        """
    }
)

print(response["output"])